# adaptive-intelligence v2 — Demo Notebook
## Self-Improving RAG with RL Routing, Vectorless Mode, and Structured Output

**Author:** [@VK_Venkatkumar](https://linkedin.com/in/venkatkumarvk)  
**Library:** [adaptive-intelligence](https://pypi.org/project/adaptive-intelligence/) | [GitHub](https://github.com/VK-Ant/adaptive-intelligence)  
**Paper:** [ResearchGate](https://www.researchgate.net/publication/405076088_Adaptive_Retrieval_Orchestration_for_Self-Learning_Knowledge_Systems)  
**Also:** [llmevalkit](https://pypi.org/project/llmevalkit/) — 61 metrics for LLM evaluation  
**Portfolio:** [vk-ant.github.io/Venkatkumar](https://vk-ant.github.io/Venkatkumar)

---

### What This Notebook Demonstrates

| Feature | Traditional RAG | Adaptive Intelligence v2 |
|---|---|---|
| Retrieval | Static vector search | RL-learned routing |
| Graph | None | Conditional activation |
| Output | Text only | JSON, CSV, DataFrame |
| Feedback | None | Thumbs up/down → RL update |
| Vector DB | Required | Optional (vectorless mode) |
| Learning | None | Improves every query |


## 1. Setup

In [ ]:
%%capture
!pip install adaptive-intelligence chromadb openai


In [ ]:
import os, json, time, re
from typing import List, Dict, Any
from collections import Counter
from google.colab import userdata

# Auto-detect API key from Colab Secrets
API_KEY = None
BASE_URL = None
MODEL = None

try:
    API_KEY = userdata.get('GROQ_API_KEY')
    BASE_URL = 'https://api.groq.com/openai/v1'
    MODEL = 'llama-3.3-70b-versatile'
    print(f'Using Groq: {MODEL}')
except Exception: pass

if not API_KEY:
    try:
        API_KEY = userdata.get('XAI_API_KEY')
        BASE_URL = 'https://api.x.ai/v1'
        MODEL = 'grok-3-mini'
        print(f'Using Grok: {MODEL}')
    except Exception: pass

if not API_KEY:
    try:
        API_KEY = userdata.get('OPENAI_API_KEY')
        BASE_URL = 'https://api.openai.com/v1'
        MODEL = 'gpt-4o-mini'
        print(f'Using OpenAI: {MODEL}')
    except Exception: pass

if not API_KEY:
    print('No API key found. Add GROQ_API_KEY, XAI_API_KEY, or OPENAI_API_KEY in Colab Secrets.')
    print('Get free key: https://console.groq.com/keys')
else:
    print(f'Base URL: {BASE_URL}')


## 2. Create Test Documents

In [ ]:
DOCUMENTS = {
    "financial_q3.txt": """NovaTech Industries Q3 2025 Report\nRevenue: $847 million (12.3% YoY increase). Product revenue $612M (15.1% up), services $235M (5.8% up).\nAmericas $510M (60.2%), EMEA $220M (26.0%), APAC $117M (13.8%).\nOperating income $127M, margin 15.0% (vs 13.2% Q3 2024). Net income $98M, $2.15/share.\nEBITDA $168M, margin 19.8%. Free cash flow $108M. Cash $1.2B.\nQ4 guidance: $870-890M revenue, 15.5-16.0% margin.""",

    "financial_q2.txt": """NovaTech Industries Q2 2025 Report\nRevenue: $798 million (9.7% YoY). Product $570M, services $228M.\nOperating income $110M, margin 13.8%. Net income $84M, $1.84/share. EBITDA $149M.\nSupply chain disruptions in APAC. 3-week delay from Meridian Semiconductors.\nR&D spending $95M (11.9% of revenue), up from $82M, for AI analytics and edge computing.""",

    "risk_assessment.txt": """NovaTech Risk Assessment 2025\nSupply Chain (HIGH): 65% dependency on Meridian Semiconductors. Secondary sourcing with Pacific Chip Alliance to reduce to 45% by Q2 2026.\nCybersecurity (MEDIUM): 3 attempted intrusions in Q2. CyberShield Partners provides managed security. $12M zero-trust investment.\nRegulatory (MEDIUM): EU AI Act compliance needed for EMEA products. Cost $8-12M.\nMarket (MEDIUM): Competition from AscentTech Solutions and Vertex Digital. Market share declined 28% to 26%.\nTalent (LOW): Retention improved to 92% from 88%.""",

    "corporate_structure.txt": """NovaTech Corporate Structure\nBoard: Sarah Chen (CEO), Marcus Thompson (CFO), Dr. Anika Patel (CTO), James Rodriguez (COO).\nSubsidiaries: CloudBridge Solutions (100%, cloud, acquired 2023 $340M), DataStream Analytics (75%, JV with Apex Capital), SecureNet Systems (60%, with CyberShield Partners).\nSuppliers: Meridian Semiconductors (chips, 65%), GlobalFab Manufacturing (PCB), TechLogistics (distribution). Pacific Chip Alliance is secondary supplier.\nCompetitors: AscentTech Solutions (analytics), Vertex Digital (cloud), Quantum Dynamics (AI).""",

    "operations.txt": """NovaTech Operations H1 2025\nAustin TX facility expanded, 18% capacity increase. Edge computing production line. Yield rate 97.2% (from 95.1%).\nPacific Chip Alliance test orders: 50,000 units, 99.1% quality pass rate. Full transition Q2 2026.\nNovaStar Edge Platform: 340 enterprise customers, 4.6/5.0 satisfaction, 2.3M events/second.\nHeadcount 12,400. Added 800 engineers, 200 AI/ML. EduTech Foundation trained 150 in AI.\nCarbon emissions down 22%. Renewable energy 68% (from 51%).""",
}

DOC_DIR = '/content/novatech_docs'
os.makedirs(DOC_DIR, exist_ok=True)
for name, content in DOCUMENTS.items():
    with open(os.path.join(DOC_DIR, name), 'w') as f:
        f.write(content)
print(f'{len(DOCUMENTS)} documents created')


## 3. Test Queries (20 across 5 categories)

In [ ]:
TEST_QUERIES = [
    {"query": "What was NovaTech total revenue in Q3 2025?", "cat": "factual", "keywords": ["847", "million"]},
    {"query": "What is the EBITDA margin for Q3?", "cat": "factual", "keywords": ["19.8"]},
    {"query": "How many employees does NovaTech have?", "cat": "factual", "keywords": ["12,400", "12400"]},
    {"query": "What is the manufacturing yield rate?", "cat": "factual", "keywords": ["97.2"]},
    {"query": "How is Meridian connected to supply chain risk?", "cat": "relational", "keywords": ["65%", "chip", "dependency"]},
    {"query": "What is CyberShield Partners relationship with NovaTech?", "cat": "relational", "keywords": ["managed security", "SecureNet"]},
    {"query": "How does Pacific Chip Alliance affect supply chain?", "cat": "relational", "keywords": ["secondary", "45%"]},
    {"query": "What entities are connected to DataStream Analytics?", "cat": "relational", "keywords": ["Apex Capital", "75%"]},
    {"query": "What are the top risk factors and mitigations?", "cat": "analytical", "keywords": ["supply chain", "cybersecurity"]},
    {"query": "Analyze operational improvements in H1 2025", "cat": "analytical", "keywords": ["capacity", "yield"]},
    {"query": "What is the competitive landscape?", "cat": "analytical", "keywords": ["AscentTech", "Vertex"]},
    {"query": "Assess R&D investment strategy", "cat": "analytical", "keywords": ["AI", "95"]},
    {"query": "Compare financial performance Q2 vs Q3 2025", "cat": "comparative", "keywords": ["798", "847"]},
    {"query": "How did operating margin change Q2 to Q3?", "cat": "comparative", "keywords": ["13.8", "15.0"]},
    {"query": "Compare product vs services revenue growth in Q3", "cat": "comparative", "keywords": ["15.1", "5.8"]},
    {"query": "Carbon emissions and renewable energy change YoY?", "cat": "comparative", "keywords": ["22%", "68%"]},
    {"query": "Meridian risk to Q2 impact to Pacific Chip mitigation to Q4 guidance?", "cat": "multi_hop", "keywords": ["delay", "Pacific Chip"]},
    {"query": "How do subsidiaries position NovaTech against competitors?", "cat": "multi_hop", "keywords": ["CloudBridge", "DataStream"]},
    {"query": "Trace: Meridian dependency to supply disruption to mitigation to Q4 outlook", "cat": "multi_hop", "keywords": ["65%", "870", "890"]},
    {"query": "Full AI strategy chain: R&D, product launch, training, compliance?", "cat": "multi_hop", "keywords": ["95", "NovaStar", "EduTech"]},
]
print(f'{len(TEST_QUERIES)} queries across {len(set(q["cat"] for q in TEST_QUERIES))} categories')


## 4. Traditional RAG (Baseline)

In [ ]:
from openai import OpenAI
import chromadb

class TraditionalRAG:
    def __init__(self, api_key, model, base_url):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model = model
        self.chroma = chromadb.Client()
        self.collection = self.chroma.get_or_create_collection('trad_rag', metadata={'hnsw:space': 'cosine'})
        self.chunks = []

    def ingest(self, doc_dir):
        cid = 0
        for fname in sorted(os.listdir(doc_dir)):
            with open(os.path.join(doc_dir, fname)) as f:
                words = f.read().split()
            for i in range(0, len(words), 150):
                chunk = ' '.join(words[i:i+150])
                self.chunks.append({'id': f'c{cid}', 'text': chunk, 'source': fname})
                cid += 1
        self.collection.add(
            ids=[c['id'] for c in self.chunks],
            documents=[c['text'] for c in self.chunks],
            metadatas=[{'source': c['source']} for c in self.chunks],
        )
        print(f'[Traditional RAG] {len(self.chunks)} chunks')

    def ask(self, query, top_k=5):
        start = time.time()
        r = self.collection.query(query_texts=[query], n_results=min(top_k, len(self.chunks)),
                                  include=['documents', 'metadatas'])
        ctx = '\n---\n'.join(r['documents'][0]) if r['documents'] else ''
        try:
            resp = self.client.chat.completions.create(
                model=self.model, temperature=0.1, max_tokens=1024,
                messages=[{'role': 'system', 'content': 'Answer from context. Cite sources.'},
                          {'role': 'user', 'content': f'Context:\n{ctx}\n\nQuestion: {query}'}])
            answer = resp.choices[0].message.content
        except Exception as e:
            answer = f'Error: {e}'
        return {'answer': answer, 'latency': time.time()-start, 'strategy': 'vector_only'}

print('Traditional RAG ready')


## 5. Adaptive Intelligence v2

In [ ]:
from adaptive_intelligence import AdaptiveAI

adaptive = AdaptiveAI(
    llm_backend='openai',
    llm_model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    domain='financial',
    storage_dir='/content/adaptive_state',
    log_level='WARNING',
)
print(f'Adaptive Intelligence v2 ready: {MODEL}')


## 6. Ingest Documents

In [ ]:
trad = TraditionalRAG(API_KEY, MODEL, BASE_URL)
trad.ingest(DOC_DIR)
print()
stats = adaptive.ingest(DOC_DIR)
print(f'[Adaptive] {stats.successful} docs, {stats.total_chunks} chunks, '
      f'{adaptive.graph.node_count} graph nodes')


## 7. Run Comparison (20 Queries)

In [ ]:
def check_coverage(answer, keywords):
    a = answer.lower()
    return sum(1 for k in keywords if k.lower() in a) / len(keywords) if keywords else 0

trad_results, adapt_results = [], []

for i, q in enumerate(TEST_QUERIES):
    print(f'[{i+1}/20] ({q["cat"]}) {q["query"][:60]}...')
    
    t = trad.ask(q['query'])
    tc = check_coverage(t['answer'], q['keywords'])
    trad_results.append({'query': q['query'], 'cat': q['cat'], 'coverage': tc, **t})
    
    a = adaptive.ask(q['query'])
    ac = check_coverage(a.answer, q['keywords'])
    adapt_results.append({'query': q['query'], 'cat': q['cat'], 'coverage': ac,
        'answer': a.answer, 'strategy': a.retrieval_strategy,
        'confidence': a.confidence,
        'faithfulness': a.evaluation.faithfulness if a.evaluation else 0,
        'graph': a.retrieval_info.graph_activated if a.retrieval_info else False})
    
    te = 'pass' if tc >= 0.5 else 'MISS'
    ae = 'pass' if ac >= 0.5 else 'MISS'
    g = ' graph' if adapt_results[-1].get('graph') else ''
    print(f'  Trad: {tc:.0%} ({te}) | Adaptive: {ac:.0%} ({ae}) [{a.retrieval_strategy}{g}]')
    time.sleep(1)

print('\nDone!')


## 8. Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cats = ['factual', 'relational', 'analytical', 'comparative', 'multi_hop']
labels = ['Factual', 'Relational', 'Analytical', 'Comparative', 'Multi-hop']

def avg(results, cat):
    vals = [r['coverage'] for r in results if r['cat'] == cat]
    return sum(vals)/len(vals) if vals else 0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(cats))
w = 0.35
t_vals = [avg(trad_results, c) for c in cats]
a_vals = [avg(adapt_results, c) for c in cats]

ax1.bar(x-w/2, t_vals, w, label='Traditional RAG', color='#E74C3C', alpha=0.8)
ax1.bar(x+w/2, a_vals, w, label='Adaptive Intelligence', color='#2ECC71', alpha=0.8)
ax1.set_ylabel('Keyword Coverage')
ax1.set_title('Answer Quality by Category')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=9)
ax1.set_ylim(0, 1.15)
ax1.legend()
for i, (tv, av) in enumerate(zip(t_vals, a_vals)):
    ax1.text(i-w/2, tv+0.02, f'{tv:.0%}', ha='center', fontsize=8)
    ax1.text(i+w/2, av+0.02, f'{av:.0%}', ha='center', fontsize=8)

confs = [r['confidence'] for r in adapt_results]
rolling = [sum(confs[max(0,i-4):i+1])/len(confs[max(0,i-4):i+1]) for i in range(len(confs))]
ax2.scatter(range(1, 21), confs, color='#2ECC71', alpha=0.5, s=40)
ax2.plot(range(1, 21), rolling, color='#F39C12', linewidth=2)
ax2.set_xlabel('Query #')
ax2.set_ylabel('Confidence')
ax2.set_title('Learning Curve')
warmup = adaptive.config.rl.warmup_queries
if warmup <= 20:
    ax2.axvline(x=warmup, color='red', linestyle='--', alpha=0.5)
    ax2.text(warmup+0.3, max(confs)*0.95, 'RL active', color='red', fontsize=8)

plt.tight_layout()
plt.show()


## 9. Strategy Analysis

In [ ]:
print('TRADITIONAL RAG: 1 strategy (vector_only) for all 20 queries')
print()
print('ADAPTIVE INTELLIGENCE:')
strats = Counter(r['strategy'] for r in adapt_results)
for s, c in sorted(strats.items(), key=lambda x: -x[1]):
    print(f'  {s}: {c}/20 ({c/20:.0%})')
graph_count = sum(1 for r in adapt_results if r.get('graph'))
print(f'  Graph activated: {graph_count}/20')
print(f'  Strategies used: {len(strats)}')


## 10. v2 Demo: Vectorless Mode

In [ ]:
# Vectorless — no ChromaDB, no embeddings, pure BM25 + graph + RL
engine_vl = AdaptiveAI(
    vectorless=True,
    llm_backend='openai', llm_model=MODEL,
    api_key=API_KEY, base_url=BASE_URL,
    domain='financial',
    storage_dir='/content/vectorless_state',
    log_level='WARNING',
)
engine_vl.ingest(DOC_DIR)

resp = engine_vl.ask('What is the Q3 revenue?')
print(f'Answer: {resp.answer[:200]}...')
print(f'Confidence: {resp.confidence:.0%}')
print(f'Strategy: {resp.retrieval_strategy}')
for c in resp.citations[:3]:
    page = f', Page {c.page}' if c.page else ''
    print(f'  Source: {c.source_document}{page}')


## 11. v2 Demo: Structured Output

In [ ]:
# JSON output
resp = adaptive.ask('Extract Q3 financial metrics', output_format='json')
print('JSON output:')
print(resp.answer[:300])
if resp.structured:
    print(f'\nParsed: {type(resp.structured).__name__}')
    print(resp.structured)


## 12. v2 Demo: User Feedback

In [ ]:
resp = adaptive.ask('What are the key risks?')
print(f'Answer: {resp.answer[:200]}...')
print(f'Confidence: {resp.confidence:.0%}')

# Thumbs up — boosts RL reward for this strategy
adaptive.feedback(resp.query_id, 'good')
print('Feedback: good (RL reward boosted)')


## 13. Dashboard

In [ ]:
print(adaptive.dashboard())
print()
print('RL Policy:')
s = adaptive.rl.get_stats()
print(f'  Warmup: {s["is_warmup"]}')
print(f'  Arms: {s["total_arms"]}')
print(f'  Exploration: {s["exploration_rate"]:.1%}')
print()
print(adaptive.memory.get_learning_summary())


## 14. Results Summary

In [ ]:
t_avg = sum(r['coverage'] for r in trad_results) / len(trad_results)
a_avg = sum(r['coverage'] for r in adapt_results) / len(adapt_results)
print(f'Overall: Traditional {t_avg:.0%} vs Adaptive {a_avg:.0%} ({(a_avg-t_avg)/max(t_avg,0.01):+.0%})')
print()
for cat, label in zip(cats, labels):
    tc = avg(trad_results, cat)
    ac = avg(adapt_results, cat)
    print(f'  {label:12s}: Trad {tc:.0%} vs Adaptive {ac:.0%}')


---
**adaptive-intelligence v2** | [PyPI](https://pypi.org/project/adaptive-intelligence/) | [GitHub](https://github.com/VK-Ant/adaptive-intelligence) | [Paper](https://www.researchgate.net/publication/405076088)  
**Also:** [llmevalkit](https://pypi.org/project/llmevalkit/) — 61 metrics for LLM evaluation  
**Author:** Venkatkumar Rajan | [@VK_Venkatkumar](https://linkedin.com/in/venkatkumarvk)
